## Campus Recruitment Salary Prediction

Given *data about campus recruitment*, let's try to predict the **salary** offered to a given student.

We will use linear regression model with L2 regularization to make our predictions.

Data source: https://www.kaggle.com/datasets/benroshan/factors-affecting-campus-placement

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import Ridge

In [2]:
data = pd.read_csv('archive/Placement_Data_Full_Class.csv')
data

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,211,M,80.60,Others,82.00,Others,Commerce,77.60,Comm&Mgmt,No,91.0,Mkt&Fin,74.49,Placed,400000.0
211,212,M,58.00,Others,60.00,Others,Science,72.00,Sci&Tech,No,74.0,Mkt&Fin,53.62,Placed,275000.0
212,213,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.0,Mkt&Fin,69.72,Placed,295000.0
213,214,F,74.00,Others,66.00,Others,Commerce,58.00,Comm&Mgmt,No,70.0,Mkt&HR,60.23,Placed,204000.0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    object 
 2   ssc_p           215 non-null    float64
 3   ssc_b           215 non-null    object 
 4   hsc_p           215 non-null    float64
 5   hsc_b           215 non-null    object 
 6   hsc_s           215 non-null    object 
 7   degree_p        215 non-null    float64
 8   degree_t        215 non-null    object 
 9   workex          215 non-null    object 
 10  etest_p         215 non-null    float64
 11  specialisation  215 non-null    object 
 12  mba_p           215 non-null    float64
 13  status          215 non-null    object 
 14  salary          148 non-null    float64
dtypes: float64(6), int64(1), object(8)
memory usage: 25.3+ KB


### Preprocessing

In [4]:
data

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,211,M,80.60,Others,82.00,Others,Commerce,77.60,Comm&Mgmt,No,91.0,Mkt&Fin,74.49,Placed,400000.0
211,212,M,58.00,Others,60.00,Others,Science,72.00,Sci&Tech,No,74.0,Mkt&Fin,53.62,Placed,275000.0
212,213,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.0,Mkt&Fin,69.72,Placed,295000.0
213,214,F,74.00,Others,66.00,Others,Commerce,58.00,Comm&Mgmt,No,70.0,Mkt&HR,60.23,Placed,204000.0


In [5]:
{column: list(data[column].unique()) for column in data.select_dtypes('object').columns}

{'gender': ['M', 'F'],
 'ssc_b': ['Others', 'Central'],
 'hsc_b': ['Others', 'Central'],
 'hsc_s': ['Commerce', 'Science', 'Arts'],
 'degree_t': ['Sci&Tech', 'Comm&Mgmt', 'Others'],
 'workex': ['No', 'Yes'],
 'specialisation': ['Mkt&HR', 'Mkt&Fin'],
 'status': ['Placed', 'Not Placed']}

In [6]:
def binary_encode(df, column_dict):
    df = df.copy()
    for column, positive_value in column_dict.items():
        df[column] = df[column].apply(lambda x: 1 if x == positive_value else 0)
    return df
    
def onehot_encode(df, column_dict):
    df = df.copy()
    for column, prefix in column_dict.items():
        dummies = pd.get_dummies(df[column], prefix=prefix, dtype=int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    return df

In [7]:
# Encode categorical features
binary_feature_dict = {
    'gender': 'M',
    'ssc_b': 'Central',
    'hsc_b': 'Central',
    'workex': 'Yes',
    'specialisation': 'Mkt&Fin',
    'status': 'Placed'
}

nominal_feature_dict = {
    'hsc_s': 'hsc',
    'degree_t': 'dt'
}

data = binary_encode(data, binary_feature_dict)

data = onehot_encode(data, nominal_feature_dict)

In [8]:
data

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,degree_p,workex,etest_p,specialisation,mba_p,status,salary,hsc_Arts,hsc_Commerce,hsc_Science,dt_Comm&Mgmt,dt_Others,dt_Sci&Tech
0,1,1,67.00,0,91.00,0,58.00,0,55.0,0,58.80,1,270000.0,0,1,0,0,0,1
1,2,1,79.33,1,78.33,0,77.48,1,86.5,1,66.28,1,200000.0,0,0,1,0,0,1
2,3,1,65.00,1,68.00,1,64.00,0,75.0,1,57.80,1,250000.0,1,0,0,1,0,0
3,4,1,56.00,1,52.00,1,52.00,0,66.0,0,59.43,0,NaN,0,0,1,0,0,1
4,5,1,85.80,1,73.60,1,73.30,0,96.8,1,55.50,1,425000.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,211,1,80.60,0,82.00,0,77.60,0,91.0,1,74.49,1,400000.0,0,1,0,1,0,0
211,212,1,58.00,0,60.00,0,72.00,0,74.0,1,53.62,1,275000.0,0,0,1,0,0,1
212,213,1,67.00,0,67.00,0,73.00,1,59.0,1,69.72,1,295000.0,0,1,0,1,0,0
213,214,0,74.00,0,66.00,0,58.00,0,70.0,0,60.23,1,204000.0,0,1,0,1,0,0


In [9]:
# Split missing salary data from the data and save it for later
missing_salaries = data.loc[data.isna().sum(axis=1) > 0, :]
missing_salaries = missing_salaries.drop('salary', axis=1)

missing_salary_ids = missing_salaries['sl_no'].reset_index(drop=True).copy()

data = data.drop(missing_salaries.index, axis=0).reset_index(drop=True)

In [10]:
missing_salary_ids

0       4
1       6
2       7
3      10
4      13
     ... 
62    199
63    202
64    207
65    209
66    215
Name: sl_no, Length: 67, dtype: int64

In [11]:
data

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,degree_p,workex,etest_p,specialisation,mba_p,status,salary,hsc_Arts,hsc_Commerce,hsc_Science,dt_Comm&Mgmt,dt_Others,dt_Sci&Tech
0,1,1,67.00,0,91.00,0,58.00,0,55.0,0,58.80,1,270000.0,0,1,0,0,0,1
1,2,1,79.33,1,78.33,0,77.48,1,86.5,1,66.28,1,200000.0,0,0,1,0,0,1
2,3,1,65.00,1,68.00,1,64.00,0,75.0,1,57.80,1,250000.0,1,0,0,1,0,0
3,5,1,85.80,1,73.60,1,73.30,0,96.8,1,55.50,1,425000.0,0,1,0,1,0,0
4,8,1,82.00,1,64.00,1,66.00,1,67.0,1,62.14,1,252000.0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,210,1,62.00,1,72.00,1,65.00,0,67.0,1,56.49,1,216000.0,0,1,0,1,0,0
144,211,1,80.60,0,82.00,0,77.60,0,91.0,1,74.49,1,400000.0,0,1,0,1,0,0
145,212,1,58.00,0,60.00,0,72.00,0,74.0,1,53.62,1,275000.0,0,0,1,0,0,1
146,213,1,67.00,0,67.00,0,73.00,1,59.0,1,69.72,1,295000.0,0,1,0,1,0,0


In [12]:
data.isna().sum(axis=0).sum(axis=0)

np.int64(0)

In [13]:
# Drop sl_no columns
data = data.drop('sl_no', axis=1)

missing_salaries = missing_salaries.drop('sl_no', axis=1)

In [14]:
# Split df into X and y
y = data['salary'].copy()
X = data.drop('salary', axis=1).copy()

In [15]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=123)

In [16]:
# Scale X with a standard scaler
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns, index = X_test.index)

missing_salaries = pd.DataFrame(scaler.transform(missing_salaries), columns = missing_salaries.columns, index = missing_salaries.index)

In [17]:
missing_salaries

,gender,ssc_p,ssc_b,hsc_p,hsc_b,degree_p,workex,etest_p,specialisation,mba_p,status,hsc_Arts,hsc_Commerce,hsc_Science,dt_Comm&Mgmt,dt_Others,dt_Sci&Tech
3,0.689202,-1.759545,0.950382,-1.932395,1.320676,-2.589951,-0.902986,-0.529642,-1.345185,-0.477883,-1.0,-0.231455,-1.052209,1.166190,-1.604917,-0.161515,1.712698
5,0.689202,-1.873410,-1.052209,-2.177443,-0.757188,-0.238989,1.107436,-1.332767,0.743392,-1.774799,-1.0,-0.231455,-1.052209,1.166190,-1.604917,-0.161515,1.712698
6,-1.450953,-2.898191,-1.052209,-2.244275,-0.757188,1.572407,-0.902986,0.074892,0.743392,-1.492286,-1.0,-0.231455,0.950382,-0.857493,0.623085,-0.161515,-0.583874
9,0.689202,-1.531816,0.950382,0.072542,1.320676,-1.202498,-0.902986,-1.405779,0.743392,-1.670716,-1.0,-0.231455,0.950382,-0.857493,0.623085,-0.161515,-0.583874
12,-1.450953,-2.784327,0.950382,-1.598239,-0.757188,-0.585853,-0.902986,-0.821688,-1.345185,0.448957,-1.0,-0.231455,-1.052209,1.166190,0.623085,-0.161515,-0.583874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,-1.450953,-0.507035,0.950382,0.072542,1.320676,-0.585853,-0.902986,1.076609,-1.345185,1.592226,-1.0,-0.231455,0.950382,-0.857493,-1.604917,6.191392,-0.583874
201,0.689202,-1.964502,0.950382,-0.707156,-0.757188,-1.664983,-0.902986,0.419506,-1.345185,-0.641443,-1.0,-0.231455,-1.052209,1.166190,0.623085,-0.161515,-0.583874
206,0.689202,-3.467514,0.950382,-3.046250,1.320676,-1.356660,-0.902986,1.733711,0.743392,-1.475765,-1.0,-0.231455,-1.052209,1.166190,0.623085,-0.161515,-0.583874
208,-1.450953,-3.239785,0.950382,-1.041312,-0.757188,-0.585853,-0.902986,1.416842,-1.345185,0.098707,-1.0,-0.231455,-1.052209,1.166190,0.623085,-0.161515,-0.583874


### Training / Results

In [18]:
model = Ridge(alpha = 100.0)
model.fit(X_train, y_train)

model_r2 = model.score(X_test, y_test)

print(f"Model R^2 Score: {model_r2:.5f}")

Model R^2 Score: 0.03140


In [19]:
missing_salary_predictions = pd.Series(model.predict(missing_salaries), name='salary')
print("Potential salaries for missing target values:")
pd.concat([missing_salary_ids, missing_salary_predictions], axis=1)

Potential salaries for missing target values:


,sl_no,salary
0,4,300029.076252
1,6,295397.270069
2,7,262684.013410
3,10,273913.473268
4,13,256581.522217
...,...,...
62,199,314834.883618
63,202,274553.852116
64,207,294912.906217
65,209,269195.020570
